# AE architecture sweep

Stage 1 (capacity, 12 configs) and Stage 2 (optimisation, 9 configs at the
Stage-1 winner) at fixed latent dim 16, selected by **purged blocked
cross-validation** (`make_blocked_cv`). The held-out test block (D-O event 8 plus
non-event padding) is excluded throughout; each config is scored by its mean
validation MSE-z across folds. Selection gates on latent conditioning
(`effective_rank`) then ranks on mean val MSE-z, so a tidy latent is weighed
alongside reconstruction. The winner is refit on the full non-test pool for the CV
mean best-epoch and persisted to
`outputs/checkpoints/02_ae_arch_sweep/ae_sweep_winner.pt` together with the test
indices, so the winner notebook evaluates on the same held-out block.

Per-stage results are logged to `stage1.csv` / `stage2.csv` incrementally.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from paleoreco.data import build_prior_cube
from paleoreco.data.splits import make_blocked_cv
from paleoreco.training.cv import cross_validate, fit_on_indices, pool_indices
from paleoreco.models.autoencoder import ConvAE, count_parameters
from paleoreco.training.trainer_ae import set_seed, train
from paleoreco.eval.ae import reconstruct_split
from paleoreco.eval.shared import compute_split_metrics, latent_tidiness

/vol/bitbucket/egg25/diss/sparse_paleoclimate_field_reconstruction/paleo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 0
LATENT_DIM = 32
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32

FOLD_YEARS = 4000     # ~5 purged blocked CV folds over the non-test pool
MIN_EFF_RANK = 8      # latent-health gate: effective rank >= half the latent dim

# Stage 1: capacity grid at the v1-default optimiser settings.
BASE_CHANNELS = (8, 16, 32, 64)
DEPTHS = (2, 3, 4)
LR_STAGE1 = 1e-3
WD_STAGE1 = 1e-4

# Stage 2: optimiser grid at the Stage-1 winning (base, depth).
LRS = (3e-4, 1e-3, 3e-3)
WDS = (1e-5, 1e-4, 1e-3)

OUT_DIR = Path(REPO_ROOT) / "outputs"
CSV_DIR = OUT_DIR / "csvs" / "02_ae_arch_sweep"
CKPT_DIR = OUT_DIR / "checkpoints" / "02_ae_arch_sweep"
STAGE1_CSV = CSV_DIR / "stage1.csv"
STAGE2_CSV = CSV_DIR / "stage2.csv"
WINNER_PT = CKPT_DIR / "ae_sweep_winner.pt"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

## Data and cross-validation folds

In [3]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
valid = data["valid"]

# Purged blocked CV: the test block (event 8 + padding) is held out of every fold;
# cross_validate refits z-score stats per fold on its train ages only.
cv = make_blocked_cv(ages, fold_years=FOLD_YEARS)
test_idx = cv["test"]
pool = pool_indices(cv["folds"])
print(
    f"{len(cv['folds'])} folds; pool={pool.size} ages; "
    f"test={test_idx.size} ages [{ages[test_idx].min()}, {ages[test_idx].max()}] yr BP"
)
for i, f in enumerate(cv["folds"]):
    va = ages[f["val"]]
    print(f"  fold {i}: train={f['train'].size:3d} val={f['val'].size:3d}  "
          f"val range [{va.min()}, {va.max()}]")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\ndevice: {device}")

5 folds; pool=644 ages; test=80 ages [37325, 39300] yr BP
  fold 0: train=443 val=161  val range [29100, 33100]
  fold 1: train=476 val=128  val range [33125, 36300]
  fold 2: train=564 val= 33  val range [40325, 41125]
  fold 3: train=410 val=161  val range [41150, 45150]
  fold 4: train=443 val=161  val range [45175, 49175]

device: cuda


## Per-config training helper

In [4]:
def encode_dataset(model, dataset, device, batch_size):
    """Encode every item of a dataset to its latent code, shape (N, latent_dim)."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    codes = []
    with torch.no_grad():
        for xb in loader:
            codes.append(model.encode(xb.to(device)).cpu().numpy())
    return np.concatenate(codes, axis=0)


def train_one_config(base_channels, depth, lr, weight_decay):
    """Cross-validate one config; return mean/std metrics + mean best-epoch.

    cross_validate refits z-score stats per fold (train ages only), so the closure
    just builds a fresh ConvAE, early-stops on the fold val, and reports
    reconstruction metrics plus latent_tidiness on the encoded val codes. The
    held-out test block never enters here.
    """
    def fit_eval(train_loader, val_loader, fold_mask, fold_stats, val_ds):
        set_seed(SEED)
        model = ConvAE(latent_dim=LATENT_DIM, base_channels=base_channels, depth=depth)
        out = train(
            model, train_loader, val_loader,
            mask=fold_mask,
            lr=lr, weight_decay=weight_decay,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            device=device, seed=SEED, verbose=False, progress=False,
        )
        model.load_state_dict(out["best_state_dict"])
        truth, pred = reconstruct_split(
            model, val_ds, device=device, batch_size=BATCH_SIZE,
        )
        m = compute_split_metrics(truth, pred, fold_mask)
        tidy = latent_tidiness(encode_dataset(model, val_ds, device, BATCH_SIZE))
        return {
            "val_mse":     m["mse"],
            "val_rmse":    m["rmse"],
            "val_rrmse":     m["rrmse"],
            "val_r_squared": m["r_squared"],
            "val_E_d":       m["E_d"],
            **tidy,
            "best_epoch":     out["best_epoch"],
            "epochs_trained": out["epochs_trained"],
            "n_params":       count_parameters(model),
        }

    return cross_validate(cube, valid, cv["folds"], fit_eval, batch_size=BATCH_SIZE)

## Stage 1: capacity sweep (12 configs)

Fix `latent_dim=16`, `lr=1e-3`, `weight_decay=1e-4`. Sweep
`base_channels` × `depth` = 4 × 3 grid.

In [5]:
stage1_rows: list[dict] = []
stage1_configs = [(b, d) for b in BASE_CHANNELS for d in DEPTHS]

for i, (base, depth) in enumerate(stage1_configs, start=1):
    print(f"\n[Stage 1 {i:>2}/{len(stage1_configs)}] base={base}  depth={depth}")
    agg = train_one_config(base, depth, LR_STAGE1, WD_STAGE1)
    row = {
        "base_channels":       base,
        "depth":               depth,
        "val_mse_mean":      agg["val_mse_mean"],
        "val_mse_std":       agg["val_mse_std"],
        "val_r_squared_mean":  agg["val_r_squared_mean"],
        "val_E_d_mean":        agg["val_E_d_mean"],
        "decorr_offdiag_mean": agg["decorr_offdiag_mean"],
        "cov_cond_mean":       agg["cov_cond_mean"],
        "effective_rank_mean": agg["effective_rank_mean"],
        "dim_var_cv_mean":     agg["dim_var_cv_mean"],
        "n_params":            agg["n_params_mean"],
        "mean_best_epoch":     agg["best_epoch_mean"],
        "n_folds":             agg["n_folds"],
    }
    stage1_rows.append(row)
    pd.DataFrame(stage1_rows).to_csv(STAGE1_CSV, index=False)
    print(
        f"  val_mse={agg['val_mse_mean']:.4f}±{agg['val_mse_std']:.4f}  "
        f"R²={agg['val_r_squared_mean']:.3f}  eff_rank={agg['effective_rank_mean']:.1f}  "
        f"decorr={agg['decorr_offdiag_mean']:.3f}  params={int(agg['n_params_mean']):,}  "
        f"mean_best_epoch={agg['best_epoch_mean']:.0f}"
    )


[Stage 1  1/12] base=8  depth=2
  val_mse=0.4521±0.2111  R²=0.849  eff_rank=1.9  decorr=0.519  params=136,142  mean_best_epoch=87

[Stage 1  2/12] base=8  depth=3
  val_mse=0.5279±0.2931  R²=0.835  eff_rank=1.9  decorr=0.513  params=78,942  mean_best_epoch=111

[Stage 1  3/12] base=8  depth=4
  val_mse=0.6073±0.2854  R²=0.804  eff_rank=2.2  decorr=0.454  params=82,814  mean_best_epoch=65

[Stage 1  4/12] base=16  depth=2
  val_mse=0.2720±0.1478  R²=0.914  eff_rank=1.9  decorr=0.495  params=277,434  mean_best_epoch=153

[Stage 1  5/12] base=16  depth=3
  val_mse=0.3070±0.1916  R²=0.906  eff_rank=2.1  decorr=0.462  params=181,466  mean_best_epoch=138

[Stage 1  6/12] base=16  depth=4
  val_mse=0.4027±0.2265  R²=0.874  eff_rank=2.9  decorr=0.391  params=262,938  mean_best_epoch=81

[Stage 1  7/12] base=32  depth=2
  val_mse=0.2034±0.1220  R²=0.936  eff_rank=2.5  decorr=0.425  params=575,570  mean_best_epoch=117

[Stage 1  8/12] base=32  depth=3
  val_mse=0.2186±0.1258  R²=0.936  eff_rank

In [6]:
# Gate on latent conditioning (effective rank), then rank survivors by mean val
# MSE-z. Fall back to the full table if the gate would empty it.
stage1_df = pd.DataFrame(stage1_rows)
healthy1 = stage1_df[stage1_df["effective_rank_mean"] >= MIN_EFF_RANK]
if healthy1.empty:
    print(f"(no config reached effective_rank >= {MIN_EFF_RANK}; ranking all)")
ranked1 = (healthy1 if not healthy1.empty else stage1_df).sort_values(
    "val_mse_mean"
).reset_index(drop=True)
print(ranked1.to_string(index=False))

winner_stage1 = ranked1.iloc[0]
BASE_STAR = int(winner_stage1["base_channels"])
DEPTH_STAR = int(winner_stage1["depth"])
print(
    f"\nStage-1 winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"val_mse={winner_stage1['val_mse_mean']:.4f}  "
    f"eff_rank={winner_stage1['effective_rank_mean']:.1f}"
)

(no config reached effective_rank >= 8; ranking all)
 base_channels  depth  val_mse_mean  val_mse_std  val_r_squared_mean  val_E_d_mean  decorr_offdiag_mean  cov_cond_mean  effective_rank_mean  dim_var_cv_mean  n_params  mean_best_epoch  n_folds
            32      2      0.203425     0.121987            0.935694      0.913887             0.424757   1.846110e+10             2.505848         0.673267  575570.0            117.2        5
            64      2      0.207451     0.117465            0.936732      0.919847             0.418127   1.411084e+05             2.720646         0.710795 1234050.0             99.8        5
            64      3      0.211138     0.118595            0.937608      0.924124             0.356947   2.144717e+09             3.302125         0.702783 1292546.0             67.0        5
            32      3      0.218562     0.125752            0.935943      0.923402             0.399047   2.434601e+04             2.958058         0.649210  457362.0         

## Stage 2: optimisation sweep at the Stage-1 winner (9 configs)

Fix `(base, depth)` at `(BASE_STAR, DEPTH_STAR)`. Sweep `lr` ×
`weight_decay` = 3 × 3 grid. The `(lr=1e-3, weight_decay=1e-4)` cell
duplicates one Stage-1 run; it is re-trained for loop uniformity.

In [7]:
stage2_rows: list[dict] = []
stage2_configs = [(lr, wd) for lr in LRS for wd in WDS]

for i, (lr, wd) in enumerate(stage2_configs, start=1):
    print(
        f"\n[Stage 2 {i:>2}/{len(stage2_configs)}] "
        f"base={BASE_STAR}  depth={DEPTH_STAR}  lr={lr:.0e}  wd={wd:.0e}"
    )
    agg = train_one_config(BASE_STAR, DEPTH_STAR, lr, wd)
    row = {
        "lr":                  lr,
        "weight_decay":        wd,
        "val_mse_mean":      agg["val_mse_mean"],
        "val_mse_std":       agg["val_mse_std"],
        "val_r_squared_mean":  agg["val_r_squared_mean"],
        "val_E_d_mean":        agg["val_E_d_mean"],
        "decorr_offdiag_mean": agg["decorr_offdiag_mean"],
        "cov_cond_mean":       agg["cov_cond_mean"],
        "effective_rank_mean": agg["effective_rank_mean"],
        "dim_var_cv_mean":     agg["dim_var_cv_mean"],
        "n_params":            agg["n_params_mean"],
        "mean_best_epoch":     agg["best_epoch_mean"],
        "n_folds":             agg["n_folds"],
    }
    stage2_rows.append(row)
    pd.DataFrame(stage2_rows).to_csv(STAGE2_CSV, index=False)
    print(
        f"  val_mse={agg['val_mse_mean']:.4f}±{agg['val_mse_std']:.4f}  "
        f"R²={agg['val_r_squared_mean']:.3f}  eff_rank={agg['effective_rank_mean']:.1f}  "
        f"mean_best_epoch={agg['best_epoch_mean']:.0f}"
    )


[Stage 2  1/9] base=32  depth=2  lr=3e-04  wd=1e-05
  val_mse=0.1840±0.1131  R²=0.946  eff_rank=2.0  mean_best_epoch=190

[Stage 2  2/9] base=32  depth=2  lr=3e-04  wd=1e-04
  val_mse=0.1835±0.1132  R²=0.946  eff_rank=2.0  mean_best_epoch=197

[Stage 2  3/9] base=32  depth=2  lr=3e-04  wd=1e-03
  val_mse=0.1839±0.1131  R²=0.946  eff_rank=2.0  mean_best_epoch=197

[Stage 2  4/9] base=32  depth=2  lr=1e-03  wd=1e-05
  val_mse=0.2017±0.1222  R²=0.936  eff_rank=2.5  mean_best_epoch=127

[Stage 2  5/9] base=32  depth=2  lr=1e-03  wd=1e-04
  val_mse=0.2032±0.1211  R²=0.936  eff_rank=2.5  mean_best_epoch=119

[Stage 2  6/9] base=32  depth=2  lr=1e-03  wd=1e-03
  val_mse=0.2027±0.1235  R²=0.936  eff_rank=2.5  mean_best_epoch=120

[Stage 2  7/9] base=32  depth=2  lr=3e-03  wd=1e-05
  val_mse=0.2154±0.1229  R²=0.932  eff_rank=2.7  mean_best_epoch=73

[Stage 2  8/9] base=32  depth=2  lr=3e-03  wd=1e-04
  val_mse=0.2163±0.1299  R²=0.932  eff_rank=2.8  mean_best_epoch=81

[Stage 2  9/9] base=32  d

In [8]:
stage2_df = pd.DataFrame(stage2_rows)
healthy2 = stage2_df[stage2_df["effective_rank_mean"] >= MIN_EFF_RANK]
if healthy2.empty:
    print(f"(no config reached effective_rank >= {MIN_EFF_RANK}; ranking all)")
ranked2 = (healthy2 if not healthy2.empty else stage2_df).sort_values(
    "val_mse_mean"
).reset_index(drop=True)
print(ranked2.to_string(index=False))

winner_stage2 = ranked2.iloc[0]
LR_STAR = float(winner_stage2["lr"])
WD_STAR = float(winner_stage2["weight_decay"])
CV_MEAN_BEST_EPOCH = int(round(float(winner_stage2["mean_best_epoch"])))
print(
    f"\nFinal winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"lr={LR_STAR:.0e}  wd={WD_STAR:.0e}\n"
    f"  val_mse={winner_stage2['val_mse_mean']:.4f}  "
    f"cv_mean_best_epoch={CV_MEAN_BEST_EPOCH}"
)

(no config reached effective_rank >= 8; ranking all)
    lr  weight_decay  val_mse_mean  val_mse_std  val_r_squared_mean  val_E_d_mean  decorr_offdiag_mean  cov_cond_mean  effective_rank_mean  dim_var_cv_mean  n_params  mean_best_epoch  n_folds
0.0003       0.00010      0.183492     0.113191            0.945891      0.937067             0.480709   2.051112e+05             2.035735         0.720833  575570.0            196.8        5
0.0003       0.00100      0.183904     0.113110            0.945735      0.936987             0.481129   2.180736e+05             2.035290         0.720930  575570.0            196.6        5
0.0003       0.00001      0.184006     0.113104            0.945696      0.936932             0.480970   2.022122e+05             2.035115         0.720838  575570.0            190.2        5
0.0010       0.00001      0.201691     0.122186            0.936181      0.914734             0.423545   2.809711e+09             2.514601         0.669493  575570.0            12

## Persist winner

Refits the winning config on the full non-test pool (the union of all CV folds,
the test block excluded) for `CV_MEAN_BEST_EPOCH` epochs with no early stopping,
then writes `outputs/checkpoints/02_ae_arch_sweep/ae_sweep_winner.pt` with the
trained state_dict, config, stopping epoch, and the held-out `test_idx`. This is
the only checkpoint produced by the sweep; the winner notebook loads it and
evaluates on the same test block.

In [9]:
final_model, pool_stats, pool_mask = fit_on_indices(
    cube, valid, pool,
    build_model=lambda: ConvAE(
        latent_dim=LATENT_DIM, base_channels=BASE_STAR, depth=DEPTH_STAR,
    ),
    train_fn=train, epochs=CV_MEAN_BEST_EPOCH,
    batch_size=BATCH_SIZE, device=device,
    lr=LR_STAR, weight_decay=WD_STAR, seed=SEED,
    verbose=False, progress=True,
)
final_model.cpu()

checkpoint = {
    "state_dict": final_model.state_dict(),
    "config": {
        "latent_dim":    LATENT_DIM,
        "base_channels": BASE_STAR,
        "depth":         DEPTH_STAR,
        "in_channels":   3,
        "out_channels":  2,
        "grid_shape":    (32, 64),
    },
    "training": {
        "lr":           LR_STAR,
        "weight_decay": WD_STAR,
        "epochs":       CV_MEAN_BEST_EPOCH,
        "seed":         SEED,
        "split":        f"blocked_cv_fold_years_{FOLD_YEARS}",
        "monitor":      "val_mse_mean (CV)",
    },
    "cv_mean_best_epoch": CV_MEAN_BEST_EPOCH,
    "test_idx":           test_idx,
}
torch.save(checkpoint, WINNER_PT)
print(f"saved {WINNER_PT}")
print(f"  config:   {checkpoint['config']}")
print(f"  training: {checkpoint['training']}")
print(
    f"  refit on pool of {pool.size} ages for {CV_MEAN_BEST_EPOCH} epochs; "
    f"test held out = {test_idx.size} ages"
)

train(d=32): 100%|██████████| 197/197 [00:16<00:00, 12.09ep/s, train_mse=0.0360, s/ep=0.1]

saved /vol/bitbucket/egg25/diss/sparse_paleoclimate_field_reconstruction/outputs/checkpoints/02_ae_arch_sweep/ae_sweep_winner.pt
  config:   {'latent_dim': 32, 'base_channels': 32, 'depth': 2, 'in_channels': 3, 'out_channels': 2, 'grid_shape': (32, 64)}
  training: {'lr': 0.0003, 'weight_decay': 0.0001, 'epochs': 197, 'seed': 0, 'split': 'blocked_cv_fold_years_4000', 'monitor': 'val_mse_mean (CV)'}
  refit on pool of 644 ages for 197 epochs; test held out = 80 ages
